In [1]:
%pip install pymupdf

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import fitz  # PyMuPDF

reports = Path("../data/reports")
keywords = ["crude steel", "scope 1", "co2 intensity", "co2e per", "per tonne", "per ton"]

texts = {}  # report name -> list of page texts, reused in later steps
for pdf in sorted(reports.glob("*.pdf")):
    doc = fitz.open(pdf)
    texts[pdf.name] = [page.get_text().lower() for page in doc]
    print(f"\n{pdf.name}: {len(doc)} pages")
    for kw in keywords:
        pages = [i + 1 for i, t in enumerate(texts[pdf.name]) if kw in t]
        print(f"   {kw!r}: {len(pages)} pages -> {pages[:15]}")

/opt/anaconda3/lib/python3.14/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(



2025-26-annual-report.pdf: 476 pages
   'crude steel': 12 pages -> [76, 79, 127, 186, 189, 212, 217, 219, 222, 223, 224, 236]
   'scope 1': 22 pages -> [123, 133, 144, 158, 180, 181, 183, 184, 185, 187, 188, 189, 190, 191, 192]
   'co2 intensity': 1 pages -> [217]
   'co2e per': 0 pages -> []
   'per tonne': 0 pages -> []
   'per ton': 4 pages -> [76, 77, 78, 217]

2025-sd-report-22-april-2026-final-upload.pdf: 94 pages
   'crude steel': 9 pages -> [32, 50, 58, 59, 60, 65, 66, 67, 75]
   'scope 1': 17 pages -> [20, 21, 23, 24, 27, 50, 51, 52, 53, 58, 62, 63, 64, 66, 67]
   'co2 intensity': 0 pages -> []
   'co2e per': 3 pages -> [20, 24, 63]
   'per tonne': 10 pages -> [20, 21, 24, 51, 58, 59, 60, 63, 65, 66]
   'per ton': 10 pages -> [20, 21, 24, 51, 58, 59, 60, 63, 65, 66]

ESG Factsheet.pdf: 16 pages
   'crude steel': 4 pages -> [1, 2, 11, 12]
   'scope 1': 2 pages -> [2, 3]
   'co2 intensity': 0 pages -> []
   'co2e per': 0 pages -> []
   'per tonne': 0 pages -> []
   'per ton': 0

In [3]:
import pymupdf

renames = {
    "435_rns_2025-12-10_5addc144-7974-407c-8cba-611a200cb268.pdf": "thyssenkrupp_AR_2024-25.pdf",
    "SSAB Annual report 2025.pdf": "SSAB_AR_2025.pdf",
    "a1091887-5d3b-4cc2-b1c5-7623695cc1b5.pdf": "Nucor_10K_2025.pdf",
    "nucor-resources-SASB-index.pdf": "Nucor_SASB_index.pdf",
}
for old, new in renames.items():
    if (reports / old).exists():
        (reports / old).rename(reports / new)

keywords = ["crude steel", "raw steel", "steel production", "scope 1",
            "emission intensity", "emissions intensity", "ghg intensity",
            "co2e per", "per tonne", "per ton", "capacity"]

texts = {}
for pdf in sorted(reports.glob("*.pdf")):
    doc = pymupdf.open(pdf)
    texts[pdf.stem] = [page.get_text().lower() for page in doc]
    print(f"\n{pdf.stem}: {len(doc)} pages")
    for kw in keywords:
        pages = [i + 1 for i, t in enumerate(texts[pdf.stem]) if kw in t]
        if pages:
            print(f"   {kw!r}: {len(pages)} -> {pages[:12]}")


2025-26-annual-report: 476 pages
   'crude steel': 12 -> [76, 79, 127, 186, 189, 212, 217, 219, 222, 223, 224, 236]
   'steel production': 43 -> [4, 6, 14, 16, 53, 54, 72, 76, 77, 78, 79, 83]
   'scope 1': 22 -> [123, 133, 144, 158, 180, 181, 183, 184, 185, 187, 188, 189]
   'emissions intensity': 1 -> [158]
   'ghg intensity': 3 -> [107, 200, 201]
   'per ton': 4 -> [76, 77, 78, 217]
   'capacity': 28 -> [14, 27, 28, 32, 56, 58, 62, 65, 74, 85, 86, 88]

2025-sd-report-22-april-2026-final-upload: 94 pages
   'crude steel': 9 -> [32, 50, 58, 59, 60, 65, 66, 67, 75]
   'raw steel': 1 -> [87]
   'steel production': 15 -> [23, 24, 27, 28, 36, 50, 59, 60, 61, 62, 63, 67]
   'scope 1': 17 -> [20, 21, 23, 24, 27, 50, 51, 52, 53, 58, 62, 63]
   'emissions intensity': 3 -> [4, 23, 83]
   'co2e per': 3 -> [20, 24, 63]
   'per tonne': 10 -> [20, 21, 24, 51, 58, 59, 60, 63, 65, 66]
   'per ton': 10 -> [20, 21, 24, 51, 58, 59, 60, 63, 65, 66]
   'capacity': 14 -> [7, 17, 19, 20, 21, 23, 24, 25, 27

In [4]:
topics = {
    "production": ["crude steel", "raw steel", "steel production"],
    "emissions": ["scope 1", "scope 2"],
    "intensity": ["emission intensity", "emissions intensity", "ghg intensity",
                  "co2e per", "per tonne", "per ton"],
    "capacity": ["capacity"],
}

def score(page_text):
    return sum(any(kw in page_text for kw in kws) for kws in topics.values())

selected = {}
for name, pages in texts.items():
    scored = sorted(((score(t), i + 1) for i, t in enumerate(pages)), reverse=True)
    top = sorted(p for s, p in scored[:8] if s >= 2)
    selected[name] = top
    chars = sum(len(pages[p - 1]) for p in top)
    print(f"{name}: pages {top}  (~{chars // 4:,} tokens)")

2025-26-annual-report: pages [185, 189, 190, 201, 209, 217, 330, 331]  (~6,948 tokens)
2025-sd-report-22-april-2026-final-upload: pages [21, 23, 24, 27, 58, 63, 65, 66]  (~11,498 tokens)
ESG Factsheet: pages [2]  (~492 tokens)
Nucor_10K_2025: pages [10, 20, 34, 42, 47]  (~5,393 tokens)
Nucor_SASB_index: pages [3, 4, 5]  (~2,331 tokens)
POSCO_Sustainability_Report_2024_eng: pages [17, 23, 158, 173, 189]  (~3,635 tokens)
SSAB_AR_2025: pages [73, 79, 83, 85, 91, 92, 95, 187]  (~8,403 tokens)
keydata-ar-2025-en: pages []  (~0 tokens)
szag-esg-figures-2025-en: pages [2]  (~2,035 tokens)
thyssenkrupp_AR_2024-25: pages [70, 73, 90, 113, 118, 119, 124, 248]  (~7,279 tokens)


In [11]:
from getpass import getpass
from pathlib import Path

key = getpass("Paste your Gemini API key and press Enter: ")
Path("../.env").write_text(f"GEMINI_API_KEY={key.strip()}\n")
print("Saved to .env:", Path("../.env").exists())

Paste your Gemini API key and press Enter:  ········


Saved to .env: True


In [12]:
%pip install google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [15]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv("../.env", override=True)
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

models = [m.name for m in client.models.list() if "flash" in m.name or "pro" in m.name]
for m in models:
    print(m)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3-pro-image
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/deep-research-pro-preview-12-2025


In [16]:
from pathlib import Path
import os

p = Path("../.env")
txt = p.read_text() if p.exists() else ""
print("file exists:", p.exists())
print("starts with GEMINI_API_KEY= :", txt.startswith("GEMINI_API_KEY="))
print("key length in file:", len(txt.split("=", 1)[1].strip()) if "=" in txt else 0)
print("key length loaded in Python:", len(os.environ.get("GEMINI_API_KEY", "")))

file exists: True
starts with GEMINI_API_KEY= : True
key length in file: 53
key length loaded in Python: 53


In [17]:
from pydantic import BaseModel

MODEL = "gemini-3.8-flash"

texts_orig = {pdf.stem: [page.get_text() for page in pymupdf.open(pdf)]
              for pdf in sorted(reports.glob("*.pdf"))}

class Value(BaseModel):
    value: float | None
    unit: str | None
    year: str | None
    page: int | None
    quote: str | None

class Extraction(BaseModel):
    crude_steel_production: Value
    scope1_emissions: Value
    scope2_emissions: Value
    co2_intensity: Value
    steelmaking_capacity: Value

PROMPT_V1 = """You extract figures from a steel company's annual report.
Use ONLY the pages below. For each field, return the value exactly as reported
(no unit conversion, no calculation), its unit, the reporting year, the page number,
and a short quote (max 20 words) that contains the number.
If a field is not on these pages, return null for all its parts.

Fields:
- crude_steel_production: total crude (raw) steel produced by the group in the reporting year
- scope1_emissions: total group Scope 1 GHG emissions
- scope2_emissions: total group Scope 2 GHG emissions (market-based if both are given)
- co2_intensity: GHG or CO2 emissions per tonne of steel
- steelmaking_capacity: total crude steelmaking capacity

PAGES:
"""
print("Setup OK")

Setup OK


In [ ]:
import time

def extract_with_retry(name, models=("gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash"), tries=3):
    global MODEL
    for m in models:
        MODEL = m
        for attempt in range(1, tries + 1):
            try:
                result = extract(name)
                print(f"OK with {m} (attempt {attempt})")
                return result
            except Exception as e:
                print(f"{m}, attempt {attempt}: {str(e)[:80]}")
                time.sleep(10 * attempt)
    raise RuntimeError("All models failed, try again later")

ssab = extract_with_retry("SSAB_AR_2025")
print(ssab.model_dump_json(indent=2))

gemini-3.8-flash, attempt 1: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
gemini-3.8-flash, attempt 2: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp


In [1]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv("../.env")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"],
                      http_options={"timeout": 30000})  # 30 seconds

for m in ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash", "gemini-3.5-flash"]:
    try:
        r = client.models.generate_content(model=m, contents="Reply with just: OK")
        print(m, "->", r.text)
    except Exception as e:
        print(m, "-> ERROR:", str(e)[:100])

gemini-3.8-flash -> ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman
gemini-3.7-flash -> ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman
gemini-3.6-flash -> OK
gemini-3.5-flash -> OK


In [2]:
from pathlib import Path
import pymupdf

reports = Path("../data/reports")
texts_orig = {pdf.stem: [p.get_text() for p in pymupdf.open(pdf)]
              for pdf in sorted(reports.glob("*.pdf"))}
texts = {k: [t.lower() for t in v] for k, v in texts_orig.items()}

topics = {
    "production": ["crude steel", "raw steel", "steel production"],
    "emissions": ["scope 1", "scope 2"],
    "intensity": ["emission intensity", "emissions intensity", "ghg intensity",
                  "co2e per", "per tonne", "per ton"],
    "capacity": ["capacity"],
}

def score(page_text):
    return sum(any(kw in page_text for kw in kws) for kws in topics.values())

selected = {}
for name, pages in texts.items():
    scored = sorted(((score(t), i + 1) for i, t in enumerate(pages)), reverse=True)
    selected[name] = sorted(p for s, p in scored[:8] if s >= 2)
print(selected)

{'Nucor_10K_2025': [10, 20, 34, 42, 47], 'Nucor_SASB_index': [3, 4, 5], 'SSAB_AR_2025': [73, 79, 83, 85, 91, 92, 95, 187], 'thyssenkrupp_AR_2024-25': [70, 73, 90, 113, 118, 119, 124, 248]}


In [3]:
from pydantic import BaseModel

MODEL = "gemini-3.6-flash"

class Value(BaseModel):
    value: float | None
    unit: str | None
    year: str | None
    page: int | None
    quote: str | None

class Extraction(BaseModel):
    crude_steel_production: Value
    scope1_emissions: Value
    scope2_emissions: Value
    co2_intensity: Value
    steelmaking_capacity: Value

PROMPT_V1 = """You extract figures from a steel company's annual report.
Use ONLY the pages below. For each field, return the value exactly as reported
(no unit conversion, no calculation), its unit, the reporting year, the page number,
and a short quote (max 20 words) that contains the number.
If a field is not on these pages, return null for all its parts.

Fields:
- crude_steel_production: total crude (raw) steel produced by the group in the reporting year
- scope1_emissions: total group Scope 1 GHG emissions
- scope2_emissions: total group Scope 2 GHG emissions (market-based if both are given)
- co2_intensity: GHG or CO2 emissions per tonne of steel
- steelmaking_capacity: total crude steelmaking capacity

PAGES:
"""

def build_pages(name):
    return "\n\n".join(f"--- PAGE {p} ---\n{texts_orig[name][p - 1]}" for p in selected[name])

def extract(name):
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT_V1 + build_pages(name),
        config={"response_mime_type": "application/json",
                "response_schema": Extraction,
                "temperature": 0},
    )
    return response.parsed

ssab = extract("SSAB_AR_2025")
print(ssab.model_dump_json(indent=2))

{
  "crude_steel_production": {
    "value": 7536.0,
    "unit": "thousand tonnes",
    "year": "2025",
    "page": 187,
    "quote": "Crude steel production, thousand tonnes 7,536"
  },
  "scope1_emissions": {
    "value": 9839.0,
    "unit": "Thousand tonnes CO2e",
    "year": "2025",
    "page": 85,
    "quote": "Total Scope 1 greenhouse gas emissions 9,839"
  },
  "scope2_emissions": {
    "value": 896.0,
    "unit": "Thousand tonnes CO2e",
    "year": "2025",
    "page": 85,
    "quote": "Total Scope 2 greenhouse gas emissions, market-based method 896"
  },
  "co2_intensity": {
    "value": null,
    "unit": null,
    "year": null,
    "page": null,
    "quote": null
  },
  "steelmaking_capacity": {
    "value": null,
    "unit": null,
    "year": null,
    "page": null,
    "quote": null
  }
}


In [4]:
import json

companies = {
    "SSAB": ["SSAB_AR_2025"],
    "thyssenkrupp": ["thyssenkrupp_AR_2024-25"],
    "Nucor": ["Nucor_10K_2025", "Nucor_SASB_index"],
}

def build_pages_multi(doc_names):
    parts = []
    for doc in doc_names:
        for p in selected[doc]:
            parts.append(f"--- DOCUMENT {doc}, PAGE {p} ---\n{texts_orig[doc][p - 1]}")
    return "\n\n".join(parts)

def extract_company(doc_names):
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT_V1 + build_pages_multi(doc_names),
        config={"response_mime_type": "application/json",
                "response_schema": Extraction,
                "temperature": 0},
    )
    return response.parsed

rows = []
for company, docs in companies.items():
    result = extract_company(docs)
    for field, v in result.model_dump().items():
        rows.append({"company": company, "field": field, **v,
                     "model": MODEL, "prompt": "v1"})
    print(f"{company}: done")

out = Path("../data/extracted")
out.mkdir(parents=True, exist_ok=True)
df_v1 = pd.DataFrame(rows) if "pd" in dir() else None
import pandas as pd
df_v1 = pd.DataFrame(rows)
df_v1.to_csv(out / "extractions_v1.csv", index=False)
print(df_v1[["company", "field", "value", "unit", "year", "page"]].to_string(index=False))

SSAB: done
thyssenkrupp: done
Nucor: done
     company                  field      value                 unit year  page
        SSAB crude_steel_production     7536.0      thousand tonnes 2025 187.0
        SSAB       scope1_emissions     9839.0 Thousand tonnes CO2e 2025  85.0
        SSAB       scope2_emissions      896.0 Thousand tonnes CO2e 2025  85.0
        SSAB          co2_intensity        NaN                  NaN  NaN   NaN
        SSAB   steelmaking_capacity        NaN                  NaN  NaN   NaN
thyssenkrupp crude_steel_production        NaN                  NaN  NaN   NaN
thyssenkrupp       scope1_emissions        NaN                  NaN  NaN   NaN
thyssenkrupp       scope2_emissions        NaN                  NaN  NaN   NaN
thyssenkrupp          co2_intensity        NaN                  NaN  NaN   NaN
thyssenkrupp   steelmaking_capacity        NaN                  NaN  NaN   NaN
       Nucor crude_steel_production 21000000.0     Metric Tons Cast 2024   3.0
       Nuc

In [5]:
page = texts_orig["thyssenkrupp_AR_2024-25"][118 - 1]
i = page.lower().find("co2e per")
print(page[max(0, i - 1500): i + 500])

y requirements, the 
availability of renewable hydrogen and access to funding. Therefore, if conditions change, it may be necessary to take 
an adaptive management approach. 
Metrics and targets 
E1-4 – Targets related to climate change mitigation and adaptation 
thyssenkrupp’s GHG reduction targets are part of the groupwide decarbonization strategy and taken together are as-
sessed by SBTi as being in accordance with the 1.5-degree Celsius target of the Paris Climate Agreement. The target 
system is aligned with the SBTi methodology and follows a cross-sector and sectoral decarbonization approach (SDA). 
The methodology is based on the IPCC’s science-based climate scenarios (Intergovernmental Panel on Climate 
Change = IPCC) and on international and European reference frameworks such as the Paris Climate Agreement and 
the EU Green Deal. Stakeholder interests – especially legislative and capital market requirements – were also consid-
ered in formulating the targets. 
Target types and

In [6]:
tk = texts["thyssenkrupp_AR_2024-25"]
for kw in ["e1-6", "gross scope 1", "total ghg emissions", "crude steel", "intensity"]:
    print(f"{kw!r}: {[i + 1 for i, t in enumerate(tk) if kw in t]}")


'e1-6': [85, 90, 113, 114, 119, 120]
'gross scope 1': [90, 123, 124]
'total ghg emissions': [85, 90, 120, 123, 124]
'crude steel': [35, 52]
'intensity': [30, 31, 69, 70, 73, 89, 90, 111, 119, 120, 124, 137, 142]


In [7]:
page = texts_orig["thyssenkrupp_AR_2024-25"][123 - 1]
i = page.lower().find("gross scope 1")
print(page[max(0, i - 300): i + 1200])

missions. 
 
 
GROSS SCOPES 1, 2, 3 AND TOTAL GHG EMISSIONS 
  
  
Retrospective
Milestones and target years
  
Base year Reporting year Mid-term target
Long-term 
target 
Annual % 
target / Base 
year
Mt CO2e 
2017 / 2018 
2024 / 2025
2029 / 2030
2049 / 2050 
 
Scope 1 GHG emissions1) 
  
 
 
  
 
Gross Scope 1 GHG emissions 
24.2 
19.8
 
  
 
Gross Scope 1 GHG emissions from regulated emission trading schemes2) 
95% 
100%
 
  
 
Scope 2 GHG emissions1) 
  
 
 
  
 
Gross location-based Scope 2 GHG emissions 
0.9 
1.1
 
  
 
Gross market-based Scope 2 GHG emissions 
1.1 
1.2
 
  
 
Subtotal Scope 1 and Scope 2 market-based3) 
25.3 
21.0
15.6
2.5 
(3.2)%
Scope 3 GHG emissions1) 
  
 
 
  
 
Gross Scope 3 GHG emissions3) 
469.1 
186.0
328.4
46.9 
(2.5)%
1) Purchased goods and services 
43.4 
27.4
 
  
 
3) Fuel and energy-related Activities 
4.3 
3.9
 
  
 
4) Upstream transportation and distribution 
0.8 
0.6
 
  
 
5) Waste generated in operations 
0.2 
0.2
 
  
 
6) Business travelin

In [8]:
strong_terms = ["gross scope 1", "e1-6", "total ghg emissions", "scope 1 ghg",
                "crude steel", "raw steel", "emissions intensity", "emission intensity",
                "ghg intensity", "co2e per", "per tonne of", "per ton of"]

selected_v2 = {}
for name, pages in texts.items():
    scored = sorted(((sum(t.count(term) for term in strong_terms), i + 1)
                     for i, t in enumerate(pages)), reverse=True)
    selected_v2[name] = sorted(p for s, p in scored[:15] if s >= 1)
    chars = sum(len(pages[p - 1]) for p in selected_v2[name])
    print(f"{name}: {selected_v2[name]}  (~{chars // 4:,} tokens)")

Nucor_10K_2025: [9, 10, 11, 14, 16]  (~5,009 tokens)
Nucor_SASB_index: [3, 4, 5]  (~2,331 tokens)
SSAB_AR_2025: [7, 17, 26, 28, 30, 53, 67, 79, 80, 83, 84, 85, 87, 115, 187]  (~13,472 tokens)
thyssenkrupp_AR_2024-25: [35, 52, 69, 70, 73, 85, 90, 113, 114, 118, 119, 120, 123, 124]  (~12,148 tokens)


In [9]:
PROMPT_V2 = """You extract reported figures from a steel company's annual report or sustainability disclosure.
Use ONLY the pages below.

Rules:
1. Take the value for the most recent reporting year shown. Tables often have several columns
   (base year, previous year, reporting year, targets). Use the reporting-year column.
   Never return a base-year value, a target or a forecast.
2. Copy the number exactly as printed. Do not convert units or calculate anything.
3. Always give the unit as printed (e.g. "Mt CO2e", "thousand tonnes", "t CO2e/t crude steel").
   If the unit is only in a table or column header, take it from there.
4. Give the reporting year as printed (e.g. "2025" or "2024/2025"), the page number from the page marker,
   and a short quote (max 20 words) containing the number.
5. Use group-wide totals, not single sites or segments.
6. If a field is not on these pages, return null for all its parts. Do not guess.

Fields:
- crude_steel_production: total crude steel produced in the reporting year (US reports may say "raw steel" or "tons cast")
- scope1_emissions: total gross Scope 1 GHG emissions
- scope2_emissions: total gross Scope 2 GHG emissions, market-based if both are given
- co2_intensity: GHG or CO2 emissions per tonne of steel (not per revenue)
- steelmaking_capacity: total crude steelmaking capacity per year

PAGES:
"""

def run_extraction(prompt, selection, version):
    rows = []
    for company, docs in companies.items():
        pages = "\n\n".join(f"--- DOCUMENT {d}, PAGE {p} ---\n{texts_orig[d][p - 1]}"
                            for d in docs for p in selection[d])
        response = client.models.generate_content(
            model=MODEL, contents=prompt + pages,
            config={"response_mime_type": "application/json",
                    "response_schema": Extraction, "temperature": 0})
        for field, v in response.parsed.model_dump().items():
            rows.append({"company": company, "field": field, **v, "model": MODEL, "prompt": version})
        print(f"{company}: done")
    return pd.DataFrame(rows)

df_v2 = run_extraction(PROMPT_V2, selected_v2, "v2")
df_v2.to_csv(out / "extractions_v2.csv", index=False)
print(df_v2[["company", "field", "value", "unit", "year", "page"]].to_string(index=False))

SSAB: done
thyssenkrupp: done
Nucor: done
     company                  field      value                 unit        year  page
        SSAB crude_steel_production     7536.0      thousand tonnes        2025  53.0
        SSAB       scope1_emissions     9839.0 Thousand tonnes CO2e        2025  84.0
        SSAB       scope2_emissions      896.0 Thousand tonnes CO2e        2025  84.0
        SSAB          co2_intensity        NaN                  NaN         NaN   NaN
        SSAB   steelmaking_capacity        8.8       million tonnes        2025   7.0
thyssenkrupp crude_steel_production        9.3         million tons 2024 / 2025  52.0
thyssenkrupp       scope1_emissions       19.8              Mt CO2e 2024 / 2025 123.0
thyssenkrupp       scope2_emissions        1.2              Mt CO2e 2024 / 2025 123.0
thyssenkrupp          co2_intensity        NaN                  NaN         NaN   NaN
thyssenkrupp   steelmaking_capacity        NaN                  NaN         NaN   NaN
       Nucor

In [10]:
check = df_v2[["company", "field", "value", "unit", "year", "page", "quote"]].copy()
check["true_value"] = ""
check["true_unit"] = ""
check["true_page"] = ""
check["verdict"] = ""   # correct / wrong value / wrong year / wrong unit / missed / correctly null
check["note"] = ""

# already verified by hand on page 123
check.loc[(check.company == "thyssenkrupp") & (check.field == "scope1_emissions"),
          ["true_value", "true_unit", "true_page", "verdict"]] = [19.8, "Mt CO2e", 123, "correct"]
check.loc[(check.company == "thyssenkrupp") & (check.field == "scope2_emissions"),
          ["true_value", "true_unit", "true_page", "verdict"]] = [1.2, "Mt CO2e", 123, "correct"]

check.to_csv(out / "validation_sheet.csv", index=False)
print("Saved:", out / "validation_sheet.csv", "-", len(check), "rows")

TypeError: Invalid value '19.8' for dtype 'str'. Value should be a string or missing value, got 'float' instead.

In [11]:
check = df_v2[["company", "field", "value", "unit", "year", "page", "quote"]].copy()
for col in ["true_value", "true_unit", "true_page", "verdict", "note"]:
    check[col] = ""
# verdict options: correct / wrong value / wrong year / wrong unit / missed / correctly null

# already verified by hand on page 123
check.loc[(check.company == "thyssenkrupp") & (check.field == "scope1_emissions"),
          ["true_value", "true_unit", "true_page", "verdict"]] = ["19.8", "Mt CO2e", "123", "correct"]
check.loc[(check.company == "thyssenkrupp") & (check.field == "scope2_emissions"),
          ["true_value", "true_unit", "true_page", "verdict"]] = ["1.2", "Mt CO2e", "123", "correct"]

check.to_csv(out / "validation_sheet.csv", index=False)
print("Saved:", out / "validation_sheet.csv", "-", len(check), "rows")


Saved: ../data/extracted/validation_sheet.csv - 15 rows


In [12]:
def candidates(v):
    if v == int(v):
        n = int(v)
        return [f"{n:,}", f"{n:,}".replace(",", " "), str(n)]
    return [str(v), str(v).replace(".", ",")]

for _, r in df_v2.dropna(subset=["value"]).iterrows():
    print("=" * 90)
    print(f"{r.company} | {r.field} | LLM: {r.value} {r.unit} ({r.year}), page {int(r.page)}")
    found = False
    for doc in companies[r.company]:
        if int(r.page) > len(texts_orig[doc]):
            continue
        page = texts_orig[doc][int(r.page) - 1]
        for c in candidates(r.value):
            i = page.find(c)
            if i >= 0:
                print(f"[{doc}, p{int(r.page)}] ...{page[max(0, i - 300): i + 120]}...")
                found = True
                break
        if found:
            break
    if not found:
        print("!! number NOT found on the cited page")

SSAB | crude_steel_production | LLM: 7536.0 thousand tonnes (2025), page 53
[SSAB_AR_2025, p53] ...on and shipments 
Crude steel production was 3% higher and rolling 
production 5% higher during 2025 compared to 
2024. SSAB’s steel shipments amounted to 6,364 
(6,134) thousand tonnes, an increase of 4% 
compared to 2024.  
Production and shipments
Thousand tonnes
2025
2024
Crude steel production
7,536
7,349
Rolling production
6,970
6,658
Shipments
6,364
6,134
ABOUT SSAB
 REPORTS 
OTHER
CORPORATE GOVERNANCE REPORT ...
SSAB | scope1_emissions | LLM: 9839.0 Thousand tonnes CO2e (2025), page 84
[SSAB_AR_2025, p84] ...on towards a fossil-free steel production.
Greenhouse gas emissions
Change 
compared to 
Base year
Outcome
Target
Target
Annual 
reduction
2025
2024
2023
2024
2018
2025
2033
2045
Thousand tonnes CO2e
%
Thousand 
tonnes CO2e
%
%
%
%
Scope 1 greenhouse gas emissions1)
Scope 1 greenhouse gas emissions
9,839
9,479
9,947
Where of Scope 1 emissions from regulated emission trading sc

In [13]:
verdicts = {
    ("SSAB", "crude_steel_production"): ("7536", "thousand tonnes", "53", "correct", ""),
    ("SSAB", "scope1_emissions"): ("9839", "thousand tonnes CO2e", "84", "correct", ""),
    ("SSAB", "scope2_emissions"): ("896", "thousand tonnes CO2e", "84", "correct", "market-based"),
    ("SSAB", "steelmaking_capacity"): ("8.8", "million tonnes", "7", "correct", ""),
    ("thyssenkrupp", "crude_steel_production"): ("9.3", "million tons", "52", "correct", "incl. deliveries to HKM"),
    ("Nucor", "crude_steel_production"): ("21000000", "metric tons", "3", "wrong year",
                                          "2024 value from SASB index; header says million t but number is in t"),
}
for (c, f), (tv, tu, tp, vd, note) in verdicts.items():
    m = (check.company == c) & (check.field == f)
    check.loc[m, ["true_value", "true_unit", "true_page", "verdict", "note"]] = [tv, tu, tp, vd, note]
check.to_csv(out / "validation_sheet.csv", index=False)
print("Saved verdicts:", (check.verdict != "").sum(), "of", len(check))

def show(doc, must_all, max_hits=4, width=250):
    hits = 0
    for i, t in enumerate(texts[doc]):
        if all(m in t for m in must_all):
            j = t.find(must_all[-1])
            print(f"[{doc}, p{i + 1}] ...{texts_orig[doc][i][max(0, j - width): j + width]}...\n")
            hits += 1
            if hits >= max_hits:
                break
    if hits == 0:
        print(f"[{doc}] no page contains all of {must_all}\n")

print("\n######## SSAB intensity")
show("SSAB_AR_2025", ["co2", "per tonne"])
print("\n######## thyssenkrupp intensity")
show("thyssenkrupp_AR_2024-25", ["ghg intensity"])
print("\n######## thyssenkrupp capacity")
show("thyssenkrupp_AR_2024-25", ["million t", "capacity"])
print("\n######## Nucor Scope 1 / 2 and intensity (SASB)")
show("Nucor_SASB_index", ["scope 1"], max_hits=3)
print("\n######## Nucor 2025 production and intensity (10-K)")
show("Nucor_10K_2025", ["ghg intensity"], max_hits=2)
show("Nucor_10K_2025", ["capacity", "million tons"], max_hits=2)

Saved verdicts: 8 of 15

######## SSAB intensity
[SSAB_AR_2025, p83] ...ore considered to be 
zero in the report. Energy reporting covers the 
use of electricity, purchased fuel and purchased 
heat. Use of electricity also includes external 
companies operating at SSAB sites. 
Intensity metrics 
Greenhouse gas intensity per tonne of steel 
corresponds to CO2e Scope 1 and 2 divided by 
total volume of crude steel (tonnes). The total 
volume of crude steel can be found in the Board 
of Directors’ Report under the heading 
Production.
Greenhouse gas intensity per sales...

[SSAB_AR_2025, p85] ...0
91
Scrap-based steel production, USA
SSAB Montpelier
0
202
204
SSAB Mobile
382
420
401
Other
Other sites reported
11
14
13
Total Scope 2 greenhouse gas emissions, market-based method
896
1,260
1,157
Scope 1 and 2 greenhouse gas emissions intensity per tonne of crude steel
Tonnes CO2e/tonne of crude steel
2025
2024
2023
Iron ore-based steel production, Nordic region
Scope 1 and 2 greenhouse gas e

In [14]:
more = {
    ("thyssenkrupp", "co2_intensity"): ("", "", "124", "correctly null", "only per sales (t CO2e/EUR), not per tonne of steel"),
    ("thyssenkrupp", "steelmaking_capacity"): ("", "", "6", "correctly null", "only future target (8.7-9.0 Mt shipments), no current capacity"),
    ("Nucor", "scope1_emissions"): ("4300000", "not printed (presumably t CO2e)", "3", "missed",
                                    "2024, SASB index; v1 found it; v2 dropped it because no unit is printed"),
}
for (c, f), (tv, tu, tp, vd, note) in more.items():
    m = (check.company == c) & (check.field == f)
    check.loc[m, ["true_value", "true_unit", "true_page", "verdict", "note"]] = [tv, tu, tp, vd, note]
check.to_csv(out / "validation_sheet.csv", index=False)
print("Saved verdicts:", (check.verdict != "").sum(), "of", len(check), "\n")

print("######## SSAB p85: full intensity table")
p = texts_orig["SSAB_AR_2025"][85 - 1]
i = p.find("intensity per tonne of crude steel")
print(p[i: i + 900])

print("\n######## Nucor: any Scope 2?")
for d in companies["Nucor"]:
    show(d, ["scope 2"], max_hits=2, width=200)

print("######## Nucor 10-K: capacity in tons")
show("Nucor_10K_2025", ["capacity", "tons"], max_hits=3, width=200)

Saved verdicts: 11 of 15 

######## SSAB p85: full intensity table
intensity per tonne of crude steel
Tonnes CO2e/tonne of crude steel
2025
2024
2023
Iron ore-based steel production, Nordic region
Scope 1 and 2 greenhouse gas emissions, location-based, per tonne of 
crude steel
1.77
1.73
1.74
Scope 1 and 2 greenhouse gas emissions, market-based, per tonne of 
crude steel
1.86
1.85
1.83
Scrap-based steel production, USA
Scope 1 and 2 greenhouse gas emissions, location-based, per tonne of 
crude steel
0.64
0.64
0.57
Scope 1 and 2 greenhouse gas emissions, market-based, per tonne of 
crude steel
0.48
0.59
0.51


######## Nucor: any Scope 2?
[Nucor_10K_2025] no page contains all of ['scope 2']

[Nucor_SASB_index, p4] ...sions intensity and mitigating our impact on climate change 
by becoming more energy-efficient and less carbon-intensive across all aspects of our business. Our 
focus extends to reducing Scope 1 and Scope 2 emissions, as well as Scope 3 emissions, particularly 
those assoc

In [15]:
final = {
    ("Nucor", "crude_steel_production"): ("21000000", "metric tons", "3", "correct",
        "latest available = 2024 (SASB); no 2025 figure in docs; header says million t but number is in t"),
    ("SSAB", "co2_intensity"): ("", "", "85", "correctly null",
        "only per segment (Nordic 1.86, US 0.48 market-based); group calc (9839+896)/7536 = 1.42"),
    ("Nucor", "scope2_emissions"): ("", "", "", "correctly null", "no Scope 2 figure in either document"),
    ("Nucor", "co2_intensity"): ("", "", "", "correctly null", "only qualitative statements"),
    ("Nucor", "steelmaking_capacity"): ("", "", "4-6", "correctly null",
        "only per mill type, in US short tons; no group total"),
}
for (c, f), (tv, tu, tp, vd, note) in final.items():
    m = (check.company == c) & (check.field == f)
    check.loc[m, ["true_value", "true_unit", "true_page", "verdict", "note"]] = [tv, tu, tp, vd, note]
check.to_csv(out / "validation_sheet.csv", index=False)

# v1 judged against the same ground truth
v1_verdicts = {
    ("SSAB", "crude_steel_production"): "correct", ("SSAB", "scope1_emissions"): "correct",
    ("SSAB", "scope2_emissions"): "correct", ("SSAB", "co2_intensity"): "correctly null",
    ("SSAB", "steelmaking_capacity"): "missed",
    ("thyssenkrupp", "crude_steel_production"): "missed", ("thyssenkrupp", "scope1_emissions"): "missed",
    ("thyssenkrupp", "scope2_emissions"): "missed", ("thyssenkrupp", "co2_intensity"): "correctly null",
    ("thyssenkrupp", "steelmaking_capacity"): "correctly null",
    ("Nucor", "crude_steel_production"): "correct", ("Nucor", "scope1_emissions"): "correct",
    ("Nucor", "scope2_emissions"): "correctly null", ("Nucor", "co2_intensity"): "correctly null",
    ("Nucor", "steelmaking_capacity"): "correctly null",
}
v2_verdicts = {(r.company, r.field): r.verdict for r in check.itertuples()}

def metrics(v):
    s = pd.Series(list(v.values()))
    correct, cnull = (s == "correct").sum(), (s == "correctly null").sum()
    errors = len(s) - correct - cnull
    return {"fields": len(s), "correct values": correct, "correctly null": cnull, "errors": errors,
            "accuracy %": round(100 * (correct + cnull) / len(s)),
            "values found (recall) %": round(100 * correct / (correct + errors))}

summary = pd.DataFrame({"v1": metrics(v1_verdicts), "v2": metrics(v2_verdicts)})
summary.to_csv(out / "accuracy_summary.csv")
print(summary)
print()
print(check.verdict.value_counts())

                         v1  v2
fields                   15  15
correct values            5   8
correctly null            6   6
errors                    4   1
accuracy %               73  93
values found (recall) %  56  89

verdict
correct           8
correctly null    6
missed            1
Name: count, dtype: int64
